# Financial News Sentiment Prediction using Deep Learning
**Dataset**: `sent_train.csv` (9,543 rows) + `sent_valid.csv` (2,388 rows)  
**Task**: 3-class classification → Bearish (0), Bullish (1), Neutral (2)  
**Models**: Simple RNN · LSTM · GRU

## Phase 1 — Data Loading & EDA

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Create output dir up front so EDA plots below have somewhere to save
os.makedirs('models', exist_ok=True)

# Load data (relative paths so the notebook runs on any machine)
train_df = pd.read_csv("data/sent_train.csv")
valid_df  = pd.read_csv("data/sent_valid.csv")

LABEL_MAP = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

print('Train shape:', train_df.shape)
print('Valid shape:', valid_df.shape)
print('\nColumns:', train_df.columns.tolist())
train_df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, [train_df, valid_df], ['Train', 'Validation']):
    counts = df['label'].map(LABEL_MAP).value_counts()
    colors = ['#E24B4A', '#3B9E5A', '#888780']
    ax.bar(counts.index, counts.values, color=colors, alpha=0.85)
    ax.set_title(f'{title} Set — Class Distribution', fontweight='bold')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 30, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=10)
    ax.set_ylim(0, max(counts.values) * 1.2)

plt.suptitle('Class Distribution — Train vs Validation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('models/class_distribution.png', dpi=150)
plt.show()
print('\n⚠ NOTE: Neutral dominates at ~65%. Must use class weights in loss function!')

In [ ]:
# Tweet length analysis
train_df['text_len'] = train_df['text'].str.len()
train_df['word_count'] = train_df['text'].str.split().str.len()

print('Text length statistics:')
print(train_df[['text_len', 'word_count']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['text_len'].hist(bins=50, ax=axes[0], color='#4e79a7', alpha=0.8)
axes[0].set_title('Character Length Distribution')
axes[0].axvline(train_df['text_len'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

train_df['word_count'].hist(bins=40, ax=axes[1], color='#f28e2b', alpha=0.8)
axes[1].set_title('Word Count Distribution')
axes[1].axvline(train_df['word_count'].median(), color='red', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.savefig('models/tweet_length_dist.png', dpi=150)
plt.show()

In [ ]:
# Sample tweets from each class
for label_id, label_name in LABEL_MAP.items():
    print(f'\n--- {label_name} ({label_id}) ---')
    samples = train_df[train_df['label'] == label_id]['text'].sample(3, random_state=42)
    for i, s in enumerate(samples, 1):
        print(f'  {i}. {s[:120]}...' if len(s) > 120 else f'  {i}. {s}')

## Phase 2 — Preprocessing

In [ ]:
import os
import torch
os.makedirs('models', exist_ok=True)

from preprocessing import (
    clean_text, Vocabulary, load_data, build_loaders, get_class_weights
)

# Load and clean
train_df, valid_df = load_data('data/sent_train.csv', 'data/sent_valid.csv')

# Show cleaning effect
sample_tweet = train_df['text'].iloc[0]
print('Original :', sample_tweet)
print('Cleaned  :', clean_text(sample_tweet))

In [ ]:
# Build vocabulary from TRAIN only
vocab = Vocabulary(max_size=15000)
vocab.build(train_df['clean'].tolist())

# Build dataloaders
train_loader, valid_loader = build_loaders(train_df, valid_df, vocab)

# Compute class weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')
class_weights = get_class_weights(train_df['label'].values, device)

# Verify batch shapes
X_batch, y_batch = next(iter(train_loader))
print(f'\nBatch — X: {X_batch.shape}, y: {y_batch.shape}')

## Phase 3 — RNN Models Training

In [ ]:
import pickle
from rnn_models import SimpleRNNClassifier, LSTMClassifier, GRUClassifier
from train_evaluate import train_model, full_evaluation, plot_history

import torch.nn as nn
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Save vocab for Streamlit
with open('models/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('Vocabulary saved.')

In [ ]:
# ── Model A: Simple RNN ───────────────────────────────────────────
print('Training Simple RNN...')
simplernn = SimpleRNNClassifier(vocab_size=len(vocab))
print(simplernn)

hist_rnn = train_model(
    simplernn, train_loader, valid_loader,
    criterion=criterion, device=device,
    epochs=15, lr=1e-3, patience=3,
    save_path='models/simplernn_best.pt'
)
plot_history(hist_rnn, 'Simple RNN')

In [ ]:
# ── Model B: LSTM ────────────────────────────────────────────────
print('Training LSTM...')
lstm = LSTMClassifier(vocab_size=len(vocab))
print(lstm)

hist_lstm = train_model(
    lstm, train_loader, valid_loader,
    criterion=criterion, device=device,
    epochs=15, lr=1e-3, patience=3,
    save_path='models/lstm_best.pt'
)
plot_history(hist_lstm, 'LSTM')

In [ ]:
# ── Model C: GRU ────────────────────────────────────────────────
print('Training GRU...')
gru = GRUClassifier(vocab_size=len(vocab))
print(gru)

hist_gru = train_model(
    gru, train_loader, valid_loader,
    criterion=criterion, device=device,
    epochs=15, lr=1e-3, patience=3,
    save_path='models/gru_best.pt'
)
plot_history(hist_gru, 'GRU')

## Phase 4 — Evaluation & Comparison

In [ ]:
from train_evaluate import full_evaluation, print_comparison_table, plot_comparison_bar

# Evaluate all models
results_rnn   = full_evaluation(simplernn, valid_loader, device, 'Simple RNN')
results_lstm  = full_evaluation(lstm,      valid_loader, device, 'LSTM')
results_gru   = full_evaluation(gru,       valid_loader, device, 'GRU')

all_results = [results_rnn, results_lstm, results_gru]

# Comparison table
comp_df = print_comparison_table(all_results)
comp_df.to_csv('models/rnn_comparison.csv', index=False)

In [ ]:
# Grouped bar chart
plot_comparison_bar(all_results)

## Summary & Conclusions

In [ ]:
import pandas as pd

summary = pd.read_csv('models/rnn_comparison.csv')
print('\nFINAL MODEL COMPARISON:')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best = summary.iloc[0]
print(f'\n✅ Best model: {best["Model"]} with Macro F1 = {best["Macro F1"]:.4f}')
print('\nKey findings:')
print('  1. Class imbalance (Neutral=65%) required class-weighted loss.')
print('  2. LSTM/GRU outperformed Simple RNN due to better long-range memory.')
print('  3. Macro F1 is a better metric than accuracy for imbalanced datasets.')